In [ ]:
from app.rl_calibration import HestonCalibEnv
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import EvalCallback
import numpy as np



In [ ]:
# 1. Load or simulate market prices for a set of strikes
strikes = np.linspace(80, 120, 9)
# For now, use Black–Scholes prices as proxy for market
market = [pricing_engine.price_call(100, K, 1.0, 0.05, 0.2) for K in strikes]


In [ ]:

# 2. Instantiate environment
env = HestonCalibEnv(strikes, market, s0=100, r=0.05, tau=1.0)

# 3. Wrap with evaluation callback
eval_env = HestonCalibEnv(strikes, market, s0=100, r=0.05, tau=1.0)
eval_callback = EvalCallback(
    eval_env, best_model_save_path='./logs/',
    log_path='./logs/', eval_freq=500, deterministic=True, render=False
)

# 4. Create and train agent
model = PPO('MlpPolicy', env, verbose=1)
model.learn(total_timesteps=20_000, callback=eval_callback)

# 5. Save final agent
model.save('heston_calibrator_ppo')
